In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("dataset_clinica_20261.csv", encoding="utf-8")

def corrigir_mojibake(valor):
    if isinstance(valor, str) and ("Ã" in valor or "Â" in valor):
        try:
            return valor.encode("latin1").decode("utf-8")
        except UnicodeError:
            return valor
    return valor

colunas_texto = df.select_dtypes(include="object").columns
df[colunas_texto] = df[colunas_texto].apply(lambda col: col.map(corrigir_mojibake))

df.head()

,cd_processo,id_processo,classe,assunto,magistrado,comarca,foro,vara,data_disponibilizacao,decisao
0,5B0005D030000,1002017-64.2024.8.26.0191,Procedimento do Juizado Especial Cível,Bancários,LUCIANA DO CARMO NOGUEIRA,Ferraz de Vasconcelos,Foro de Ferraz de Vasconcelos,Vara do Juizado Especial Cível e Criminal,27/08/2024,SENTENÇA\n\nProcesso Digital Nº:\t1002017-64.2...
1,03001EPSV0000,1034196-67.2023.8.26.0003,Procedimento Comum Cível,Bancários,Laura Mota Lima de Oliveira Baccin,SÃO PAULO,Foro Regional III - Jabaquara,1ª Vara Cível,27/08/2024,SENTENÇA\n\nProcesso Digital nº:\t1034196-67.2...
2,I20002VT10000,1000247-51.2023.8.26.0650,Procedimento Comum Cível,Empréstimo consignado,Marcia Yoshie Ishikawa,Valinhos,Foro de Valinhos,3ª Vara,27/08/2024,SENTENÇA\n\nProcesso Digital nº:\t1000247-51.2...
3,DE000DZ0N0000,1010110-16.2024.8.26.0482,Procedimento Comum Cível,Bancários,Leonardo Mazzilli Marcondes,Presidente Prudente,Foro de Presidente Prudente,4ª Vara Cível,27/08/2024,SENTENÇA\n\nProcesso Digital nº:\t1010110-16.2...
4,2S001VMCT0000,1101723-02.2024.8.26.0100,Procedimento Comum Cível,Bancários,MICHELLE FABIOLA DITTERT PUPULIM,SÃO PAULO,Foro Regional III - Jabaquara,6ª Vara Cível,27/08/2024,SENTENÇA\n\nProcesso Digital nº:\t1101723-02.2...


In [3]:
df.columns

Index(['cd_processo', 'id_processo', 'classe', 'assunto', 'magistrado',
       'comarca', 'foro', 'vara', 'data_disponibilizacao', 'decisao'],
      dtype='object')

In [4]:
# Padroniza os nomes de magistrado em maiúsculas
df["magistrado"] = df["magistrado"].astype(str).str.strip().str.upper()

# Defina um nome para filtrar (ex.: "LUCIANA DO CARMO NOGUEIRA") ou deixe None
magistrado_filtro = None

if magistrado_filtro:
    df_filtrado = df[df["magistrado"].str.contains(magistrado_filtro.upper(), na=False)]
else:
    df_filtrado = df

coluna_assunto = "assunto" if "assunto" in df_filtrado.columns else "assuntos"
colunas_para_analisar = ["classe", coluna_assunto, "foro", "vara", "magistrado"]

for coluna in colunas_para_analisar:
    print(f"\n=== value_counts: {coluna} ===")
    print(df_filtrado[coluna].value_counts(dropna=False))


=== value_counts: classe ===
classe
Procedimento Comum Cível                                          16652
Procedimento do Juizado Especial Cível                             4053
Cumprimento de sentença                                            1766
Cumprimento Provisório de Sentença                                   99
Ação de Exigir Contas                                                84
Produção Antecipada da Prova                                         50
Procedimento de Repactuação de Dívidas (Superendividamento)          33
Tutela Antecipada Antecedente                                        29
Tutela Cautelar Antecedente                                          15
Cumprimento Provisório de Decisão                                    14
Liquidação de Sentença pelo Procedimento Comum                       13
Embargos à Execução                                                  12
NaN                                                                  10
Liquidação por Arbitramento

In [5]:
coluna_decisao = "decisao"
coluna_processo = "id_processo" if "id_processo" in df.columns else None

top10 = df[[coluna_decisao]].head(10).copy()
if coluna_processo:
    top10.insert(0, coluna_processo, df[coluna_processo].head(10).values)

linhas = []
for i, (_, row) in enumerate(top10.iterrows(), start=1):
    if coluna_processo:
        linhas.append(f"Processo {i} - {row[coluna_processo]}")
    else:
        linhas.append(f"Processo {i}")
    linhas.append(str(row[coluna_decisao]))
    linhas.append("-" * 80)

texto_saida = "\n".join(linhas)
arquivo_saida = "top10_decisoes.txt"

with open(arquivo_saida, "w", encoding="utf-8") as f:
    f.write(texto_saida)

print(f"Arquivo salvo: {arquivo_saida}")

Arquivo salvo: top10_decisoes.txt


In [6]:
import re
import unicodedata

def normalizar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).lower()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

padrao_justica_gratuita = re.compile(
    r"\b("
    r"justica\s+gratuita|"
    r"gratuidade\s+da\s+justica|"
    r"beneficio\s+da\s+justica\s+gratuita|"
    r"assistencia\s+judiciaria\s+gratuita|"
    r"ajg"
    r")\b",
    flags=re.IGNORECASE,
 )

coluna_flag = "tem_justica_gratuita"
mascara_justica_gratuita = df["decisao"].apply(
    lambda x: bool(padrao_justica_gratuita.search(normalizar_texto(x)))
 )

if coluna_flag in df.columns:
    df[coluna_flag] = mascara_justica_gratuita
else:
    posicao = df.columns.get_loc("magistrado") + 1 if "magistrado" in df.columns else len(df.columns)
    df.insert(posicao, coluna_flag, mascara_justica_gratuita)

df[["magistrado", coluna_flag, "decisao"]].head(10)

,magistrado,tem_justica_gratuita,decisao
0,LUCIANA DO CARMO NOGUEIRA,True,SENTENÇA\n\nProcesso Digital Nº:\t1002017-64.2...
1,LAURA MOTA LIMA DE OLIVEIRA BACCIN,True,SENTENÇA\n\nProcesso Digital nº:\t1034196-67.2...
2,MARCIA YOSHIE ISHIKAWA,False,SENTENÇA\n\nProcesso Digital nº:\t1000247-51.2...
3,LEONARDO MAZZILLI MARCONDES,True,SENTENÇA\n\nProcesso Digital nº:\t1010110-16.2...
4,MICHELLE FABIOLA DITTERT PUPULIM,False,SENTENÇA\n\nProcesso Digital nº:\t1101723-02.2...
5,ALÉSSIO MARTINS GONÇALVES,False,SENTENÇA\n\nProcesso nº:\t1009437-72.2023.8.26...
6,VINICIUS RODRIGUES VIEIRA,False,Processo nº:\t0067587-64.2009.8.26.0506\nClass...
7,JOANNA TERRA SAMPAIO DOS SANTOS,False,SENTENÇA\n\nProcesso nº:\t1023563-21.2024.8.26...
8,MÔNICA DE CASSIA THOMAZ PEREZ REIS LOBO,False,SENTENÇA\n\nProcesso nº:\t1101899-78.2024.8.26...
9,MELISSA BERTOLUCCI,False,SENTENÇA\n\nProcesso Digital nº:\t1120452-76.2...


In [7]:
print(df_filtrado["tem_justica_gratuita"].value_counts(dropna=False))

tem_justica_gratuita
True     12434
False    10432
Name: count, dtype: int64


In [8]:
coluna_assunto = "assunto" if "assunto" in df.columns else "assuntos"
coluna_flag = "tem_justica_gratuita"

# Tabela por assunto x justiça gratuita
resumo_assunto = pd.crosstab(df[coluna_assunto], df[coluna_flag], dropna=False)
resumo_assunto = resumo_assunto.rename(columns={False: "nao", True: "sim"})

# Garante as colunas mesmo se uma categoria não aparecer
for c in ["sim", "nao"]:
    if c not in resumo_assunto.columns:
        resumo_assunto[c] = 0

resumo_assunto["total"] = resumo_assunto["sim"] + resumo_assunto["nao"]
resumo_assunto["pct_sim"] = (resumo_assunto["sim"] / resumo_assunto["total"] * 100).round(2)
resumo_assunto = resumo_assunto.sort_values(["sim", "total"], ascending=False)

print("Resumo por assunto (justiça gratuita: sim/nao):")
display(resumo_assunto.head(30))

print("\nAssuntos com pelo menos 1 ocorrência de JUSTIÇA GRATUITA:")
display(resumo_assunto[resumo_assunto["sim"] > 0].head(30))

print("\nAssuntos sem ocorrência de JUSTIÇA GRATUITA:")
display(resumo_assunto[resumo_assunto["sim"] == 0].head(30))

Resumo por assunto (justiça gratuita: sim/nao):


tem_justica_gratuita,nao,sim,total,pct_sim
assunto,,,,
Bancários,8502,8872,17374,51.06
Empréstimo consignado,1435,2969,4404,67.42
"Revisão de Juros Remuneratórios, Capitalização/Anatocismo",273,336,609,55.17
Crédito Direto ao Consumidor - CDC,108,128,236,54.24
Tarifas,102,121,223,54.26
Crédito Rotativo,12,8,20,40.00



Assuntos com pelo menos 1 ocorrência de JUSTIÇA GRATUITA:


tem_justica_gratuita,nao,sim,total,pct_sim
assunto,,,,
Bancários,8502,8872,17374,51.06
Empréstimo consignado,1435,2969,4404,67.42
"Revisão de Juros Remuneratórios, Capitalização/Anatocismo",273,336,609,55.17
Crédito Direto ao Consumidor - CDC,108,128,236,54.24
Tarifas,102,121,223,54.26
Crédito Rotativo,12,8,20,40.00



Assuntos sem ocorrência de JUSTIÇA GRATUITA:


tem_justica_gratuita,nao,sim,total,pct_sim
assunto,,,,


In [9]:
import re
import unicodedata

def _normalizar(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).upper()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()

def extrair_resultado_julgo(texto_decisao):
    t = _normalizar(texto_decisao)

    # Pega trecho imediatamente após JULGO/HOMOLOGO para classificar melhor
    m = re.search(r"\b(?:JULGO|HOMOLOGO)\b(.{0,220})", t)
    trecho = m.group(1) if m else t[:220]

    if re.search(r"PARCIALMENTE\s+PROCEDENTE", trecho):
        return "PARCIALMENTE PROCEDENTE"
    if re.search(r"IMPROCEDENT(?:E|ES)", trecho):
        return "IMPROCEDENTE"
    if re.search(r"PROCEDENT(?:E|ES)", trecho):
        return "PROCEDENTE"
    if re.search(r"EXTINT[AO].*SEM\s+RESOLUCAO\s+DO\s+MERITO|SEM\s+ANALISE\s+DO\s+MERITO", trecho):
        return "EXTINTO SEM RESOLUCAO DE MERITO"
    if re.search(r"EXTINT[AO]", trecho):
        return "EXTINTO"
    if re.search(r"HOMOLOGO.*DESISTENCIA|DESISTENCIA\s+DA\s+ACAO", trecho):
        return "DESISTENCIA HOMOLOGADA"
    if re.search(r"INDEFIRO\s+A\s+PETICAO\s+INICIAL", trecho):
        return "INDEFERIMENTO DA INICIAL"
    return "NAO IDENTIFICADO"

nova_coluna = "resultado_julgo"
serie_resultado = df["decisao"].apply(extrair_resultado_julgo)

if nova_coluna in df.columns:
    df[nova_coluna] = serie_resultado
else:
    pos_decisao = df.columns.get_loc("decisao") + 1 if "decisao" in df.columns else len(df.columns)
    df.insert(pos_decisao, nova_coluna, serie_resultado)

print(df[nova_coluna].value_counts(dropna=False))
df[["decisao", nova_coluna]].head(10)

resultado_julgo
EXTINTO                            6362
IMPROCEDENTE                       5921
NAO IDENTIFICADO                   3565
PARCIALMENTE PROCEDENTE            2899
PROCEDENTE                         2382
EXTINTO SEM RESOLUCAO DE MERITO    1695
DESISTENCIA HOMOLOGADA               41
INDEFERIMENTO DA INICIAL              1
Name: count, dtype: int64


,decisao,resultado_julgo
0,SENTENÇA\n\nProcesso Digital Nº:\t1002017-64.2...,IMPROCEDENTE
1,SENTENÇA\n\nProcesso Digital nº:\t1034196-67.2...,IMPROCEDENTE
2,SENTENÇA\n\nProcesso Digital nº:\t1000247-51.2...,EXTINTO
3,SENTENÇA\n\nProcesso Digital nº:\t1010110-16.2...,EXTINTO
4,SENTENÇA\n\nProcesso Digital nº:\t1101723-02.2...,NAO IDENTIFICADO
5,SENTENÇA\n\nProcesso nº:\t1009437-72.2023.8.26...,PARCIALMENTE PROCEDENTE
6,Processo nº:\t0067587-64.2009.8.26.0506\nClass...,EXTINTO
7,SENTENÇA\n\nProcesso nº:\t1023563-21.2024.8.26...,EXTINTO SEM RESOLUCAO DE MERITO
8,SENTENÇA\n\nProcesso nº:\t1101899-78.2024.8.26...,EXTINTO SEM RESOLUCAO DE MERITO
9,SENTENÇA\n\nProcesso Digital nº:\t1120452-76.2...,EXTINTO SEM RESOLUCAO DE MERITO


In [10]:
# Filtra decisões classificadas como extinto
coluna_resultado = "resultado_julgo"
mascara_extinto = df[coluna_resultado].fillna("").str.contains("EXTINTO", case=False, na=False)
df_extintos = df[mascara_extinto].copy()

print(f"Total de processos classificados como extinto: {len(df_extintos)}")
display(df_extintos[["id_processo", coluna_resultado, "assunto"]].head(20))

# Exporta as decisões extintas para .txt
linhas_extintos = []
for i, (_, row) in enumerate(df_extintos.iterrows(), start=1):
    linhas_extintos.append(f"Processo {i} - {row.get('id_processo', 'SEM_ID')}")
    linhas_extintos.append(f"Classificacao: {row.get(coluna_resultado, 'NAO IDENTIFICADO')}")
    linhas_extintos.append(str(row.get("decisao", "")))
    linhas_extintos.append("-" * 80)

arquivo_extintos = "decisoes_extintos.txt"
with open(arquivo_extintos, "w", encoding="utf-8") as f:
    f.write("\n".join(linhas_extintos))

print(f"Arquivo salvo: {arquivo_extintos}")

Total de processos classificados como extinto: 8057


,id_processo,resultado_julgo,assunto
2,1000247-51.2023.8.26.0650,EXTINTO,Empréstimo consignado
3,1010110-16.2024.8.26.0482,EXTINTO,Bancários
6,0067587-64.2009.8.26.0506,EXTINTO,Bancários
7,1023563-21.2024.8.26.0016,EXTINTO SEM RESOLUCAO DE MERITO,Tarifas
8,1101899-78.2024.8.26.0100,EXTINTO SEM RESOLUCAO DE MERITO,Bancários
9,1120452-76.2024.8.26.0100,EXTINTO SEM RESOLUCAO DE MERITO,Bancários
10,1011520-08.2024.8.26.0451,EXTINTO,Bancários
11,0014038-08.2023.8.26.0003,EXTINTO,Bancários
12,1057594-09.2024.8.26.0100,EXTINTO,Bancários
13,1029983-76.2023.8.26.0016,EXTINTO,Bancários


Arquivo salvo: decisoes_extintos.txt


In [11]:
# Filtra decisões classificadas como NAO IDENTIFICADO
coluna_resultado = "resultado_julgo"
mascara_nao_identificado = df[coluna_resultado].fillna("").str.upper().eq("NAO IDENTIFICADO")
df_nao_identificado = df[mascara_nao_identificado].copy()

print(f"Total de processos classificados como NAO IDENTIFICADO: {len(df_nao_identificado)}")
display(df_nao_identificado[["id_processo", coluna_resultado, "assunto"]].head(20))

# Exporta as decisões NAO IDENTIFICADO para .txt
linhas_nao_identificado = []
for i, (_, row) in enumerate(df_nao_identificado.iterrows(), start=1):
    linhas_nao_identificado.append(f"Processo {i} - {row.get('id_processo', 'SEM_ID')}")
    linhas_nao_identificado.append(f"Classificacao: {row.get(coluna_resultado, 'NAO IDENTIFICADO')}")
    linhas_nao_identificado.append(str(row.get("decisao", "")))
    linhas_nao_identificado.append("-" * 80)

arquivo_nao_identificado = "decisoes_nao_identificado.txt"
with open(arquivo_nao_identificado, "w", encoding="utf-8") as f:
    f.write("\n".join(linhas_nao_identificado))

print(f"Arquivo salvo: {arquivo_nao_identificado}")

Total de processos classificados como NAO IDENTIFICADO: 3565


,id_processo,resultado_julgo,assunto
4,1101723-02.2024.8.26.0100,NAO IDENTIFICADO,Bancários
16,1000282-33.2024.8.26.0405,NAO IDENTIFICADO,Bancários
17,1008116-87.2024.8.26.0405,NAO IDENTIFICADO,Bancários
22,1000294-47.2024.8.26.0405,NAO IDENTIFICADO,Bancários
33,1000197-39.2023.8.26.0322,NAO IDENTIFICADO,Empréstimo consignado
34,1018842-55.2024.8.26.0071,NAO IDENTIFICADO,Bancários
38,1006689-69.2024.8.26.0271,NAO IDENTIFICADO,Bancários
42,1013893-90.2023.8.26.0016,NAO IDENTIFICADO,Bancários
45,1000932-09.2023.8.26.0246,NAO IDENTIFICADO,Empréstimo consignado
48,1003008-90.2023.8.26.0024,NAO IDENTIFICADO,Empréstimo consignado


Arquivo salvo: decisoes_nao_identificado.txt


In [12]:
import os
import json
from openai import OpenAI

# Defina sua chave no ambiente antes de rodar:
# setx OPENAI_API_KEY "sua_chave"  (Windows)
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("Defina a variável de ambiente OPENAI_API_KEY antes de executar esta célula.")

client = OpenAI(api_key=api_key)
modelo_openai = "gpt-4.1-mini"

# Controle de custo: ajuste para None se quiser processar tudo
limite_linhas = 100
df_alvo = df.head(limite_linhas).copy() if limite_linhas else df.copy()

def extrair_valor_clausula_openai(texto_decisao):
    prompt = f"""
Você receberá o texto de uma decisão judicial em português.
Extraia o valor monetário ligado à cláusula/obrigação principal decidida no caso.

Regras:
1) Retorne APENAS JSON válido.
2) Use este formato exato:
{{
  "valor_clausula": "R$ 0,00" ou null,
  "trecho_evidencia": "trecho curto do texto" ou null,
  "resumo": "frase curta"
}}
3) Se não houver valor claro, use null em valor_clausula.
4) Não invente valores.

Texto da decisão:
{texto_decisao}
"""

    try:
        resposta = client.responses.create(
            model=modelo_openai,
            input=prompt,
            temperature=0
        )
        conteudo = resposta.output_text.strip()
        dados = json.loads(conteudo)

        return {
            "valor_clausula_openai": dados.get("valor_clausula"),
            "trecho_clausula_openai": dados.get("trecho_evidencia"),
            "resumo_clausula_openai": dados.get("resumo")
        }
    except Exception as e:
        return {
            "valor_clausula_openai": None,
            "trecho_clausula_openai": None,
            "resumo_clausula_openai": f"ERRO: {str(e)}"
        }

resultados = df_alvo["decisao"].fillna("").apply(extrair_valor_clausula_openai).apply(pd.Series)

for col in ["valor_clausula_openai", "trecho_clausula_openai", "resumo_clausula_openai"]:
    if col in df.columns:
        df.loc[df_alvo.index, col] = resultados[col]
    else:
        pos = df.columns.get_loc("decisao") + 1
        df.insert(pos, col, None)
        df.loc[df_alvo.index, col] = resultados[col]

print(f"Processadas {len(df_alvo)} decisões com OpenAI.")
display(df.loc[df_alvo.index, ["id_processo", "decisao", "valor_clausula_openai", "trecho_clausula_openai", "resumo_clausula_openai"]].head(10))

Processadas 100 decisões com OpenAI.


,id_processo,decisao,valor_clausula_openai,trecho_clausula_openai,resumo_clausula_openai
0,1002017-64.2024.8.26.0191,SENTENÇA\n\nProcesso Digital Nº:\t1002017-64.2...,"R$ 13.120,00","em 15/03/2024, foi realizado na conta corrente...",Valor do empréstimo contestado pela autora
1,1034196-67.2023.8.26.0003,SENTENÇA\n\nProcesso Digital nº:\t1034196-67.2...,None,None,Não há valor monetário específico decidido na ...
2,1000247-51.2023.8.26.0650,SENTENÇA\n\nProcesso Digital nº:\t1000247-51.2...,None,None,ERRO: Expecting value: line 1 column 1 (char 0)
3,1010110-16.2024.8.26.0482,SENTENÇA\n\nProcesso Digital nº:\t1010110-16.2...,"R$ 20.000,00","verba indenizatória, a ser arbitrada na quanti...",Valor da verba indenizatória por lesão moral p...
4,1101723-02.2024.8.26.0100,SENTENÇA\n\nProcesso Digital nº:\t1101723-02.2...,None,None,ERRO: Expecting value: line 1 column 1 (char 0)
5,1009437-72.2023.8.26.0477,SENTENÇA\n\nProcesso nº:\t1009437-72.2023.8.26...,None,None,ERRO: Expecting value: line 1 column 1 (char 0)
6,0067587-64.2009.8.26.0506,Processo nº:\t0067587-64.2009.8.26.0506\nClass...,None,None,ERRO: Expecting value: line 1 column 1 (char 0)
7,1023563-21.2024.8.26.0016,SENTENÇA\n\nProcesso nº:\t1023563-21.2024.8.26...,None,None,ERRO: Expecting value: line 1 column 1 (char 0)
8,1101899-78.2024.8.26.0100,SENTENÇA\n\nProcesso nº:\t1101899-78.2024.8.26...,None,None,ERRO: Expecting value: line 1 column 1 (char 0)
9,1120452-76.2024.8.26.0100,SENTENÇA\n\nProcesso Digital nº:\t1120452-76.2...,None,None,ERRO: Expecting value: line 1 column 1 (char 0)


In [13]:
# Exporta para Excel o recorte processado pela OpenAI
colunas_exportar = [
    "id_processo",
    "decisao",
    "valor_clausula_openai",
    "trecho_clausula_openai",
    "resumo_clausula_openai"
]

df_exportar = df.loc[df_alvo.index, colunas_exportar].copy()
arquivo_xlsx = "extracao_openai_primeiras_100.xlsx"

df_exportar.to_excel(arquivo_xlsx, index=False)
print(f"Arquivo Excel salvo: {arquivo_xlsx} | linhas: {len(df_exportar)}")

Arquivo Excel salvo: extracao_openai_primeiras_100.xlsx | linhas: 100


In [14]:
# Gera 2 arquivos .txt com 10 exemplos de decisões
coluna_resultado = "resultado_julgo"

def salvar_exemplos_resultado(df_base, resultado, nome_arquivo, n=10):
    amostra = df_base[df_base[coluna_resultado].fillna("").str.upper().eq(resultado)].head(n).copy()
    linhas_saida = []

    for i, (_, row) in enumerate(amostra.iterrows(), start=1):
        linhas_saida.append(f"Exemplo {i} - Processo: {row.get('id_processo', 'SEM_ID')}")
        linhas_saida.append(f"Classificacao: {resultado}")
        linhas_saida.append(str(row.get("decisao", "")))
        linhas_saida.append("-" * 80)

    with open(nome_arquivo, "w", encoding="utf-8") as f:
        f.write("\n".join(linhas_saida))

    print(f"Arquivo salvo: {nome_arquivo} | exemplos: {len(amostra)}")
    return amostra

amostra_procedente = salvar_exemplos_resultado(df, "PROCEDENTE", "decisoes_procedente_10.txt", n=10)
amostra_improcedente = salvar_exemplos_resultado(df, "IMPROCEDENTE", "decisoes_improcedente_10.txt", n=10)

display(amostra_procedente[["id_processo", coluna_resultado]].head(10))
display(amostra_improcedente[["id_processo", coluna_resultado]].head(10))

Arquivo salvo: decisoes_procedente_10.txt | exemplos: 10
Arquivo salvo: decisoes_improcedente_10.txt | exemplos: 10


,id_processo,resultado_julgo
23,1173715-57.2023.8.26.0100,PROCEDENTE
28,1009610-38.2022.8.26.0637,PROCEDENTE
39,1001831-93.2024.8.26.0400,PROCEDENTE
47,1001216-32.2022.8.26.0511,PROCEDENTE
51,1002021-55.2024.8.26.0562,PROCEDENTE
55,1020477-16.2021.8.26.0576,PROCEDENTE
60,1151788-35.2023.8.26.0100,PROCEDENTE
69,1011668-11.2024.8.26.0001,PROCEDENTE
82,1065578-18.2022.8.26.0002,PROCEDENTE
83,1009741-92.2023.8.26.0664,PROCEDENTE


,id_processo,resultado_julgo
0,1002017-64.2024.8.26.0191,IMPROCEDENTE
1,1034196-67.2023.8.26.0003,IMPROCEDENTE
14,1006147-37.2024.8.26.0114,IMPROCEDENTE
15,1013165-23.2023.8.26.0348,IMPROCEDENTE
20,1031675-74.2022.8.26.0007,IMPROCEDENTE
21,0004033-58.2022.8.26.0197,IMPROCEDENTE
30,1002085-89.2024.8.26.0457,IMPROCEDENTE
31,1008792-39.2024.8.26.0048,IMPROCEDENTE
32,1000921-49.2024.8.26.0244,IMPROCEDENTE
35,1112149-73.2024.8.26.0100,IMPROCEDENTE
